## Tool Use Pattern

Another core pattern from Andrew Ng's AgenticAI course. Where reflection is about improving outputs iteratively, tool use is about extending what the LLM can actually *do* — giving it access to external systems, live data, and computation it couldn't handle on its own.

The key insight from the course: the LLM doesn't execute tools itself. It decides *which tool to call* and *what parameters to pass*, and your code does the actual execution. The result comes back into the conversation and the LLM synthesizes a final answer from it.

The flow looks like this:

```
User query → LLM → tool call (name + params) → your code executes it → result back to LLM → final answer
```

The LLM never runs the tool — it just outputs a structured request. Your application intercepts that, runs the actual function, and feeds the result back into the conversation as a new message. The LLM then uses that result to compose its response.

This is what makes the message history important. After a tool call, you need to append three things:
1. The assistant message containing the `tool_calls` request
2. A `tool` role message with the result and the matching `tool_call_id`
3. Then call the LLM again to get the final answer

If you skip any of these or get the order wrong, the model loses context about what it called and why.

### What you define tools with

Tools are described to the LLM using a JSON schema. Each tool has a name, a description of what it does and when to use it, and a `parameters` object describing the inputs. The description is not just documentation — the model reads it to decide whether to call that tool. A vague description leads to wrong tool selection.

```python
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city. Use this when the user asks about weather conditions or temperature in a specific location.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name, e.g. 'London'"},
                "units": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}
            },
            "required": ["city"]
        }
    }
}
```

### Common categories

- **Information retrieval** — web search, database queries, RAG lookups
- **Computation** — calculators, code interpreters, data analysis
- **External services** — weather APIs, calendar, email, payment systems
- **Domain-specific** — stock prices, translation, image analysis

### Use Cases

The clearest use case is anything involving real-time or live data — current weather, stock prices, today's news. The model's training has a cutoff date, so for anything that changes it needs a tool.

The second big category is computation. LLMs are unreliable at arithmetic and precise calculations — even simple ones. Rather than having the model guess, you hand off to a calculator or code interpreter and get an exact answer back.

The third is database or document retrieval. Instead of hoping the model knows the answer from training, you query your actual data and inject the result into the conversation. This is also the foundation of RAG.

Beyond those, tool use unlocks task automation — scheduling, sending messages, triggering workflows. The model becomes an orchestrator that understands intent and invokes the right system rather than just generating text about what should happen.

### Benefits and Limitations

The main benefit is grounding. Tool results are facts, not model-generated guesses. When the calculator returns 13.125, that's correct — the model isn't estimating it. This directly reduces hallucinations for any fact that can be looked up or computed.

It's also how you give a model access to your private data without retraining. Your database, your documents, your systems — expose them as tools and the model can query them on demand.

The limitations are practical. Every tool call adds latency (a network round-trip or function execution). Tool calls also consume tokens — the tool definition, the call itself, and the result all go into the context window. For large results (a long database query, a big search response), you may need to truncate or summarize before feeding back to the model.

The other risk is error handling. Tools fail — APIs go down, queries return nothing, parameters get misformatted. The model needs to receive a clear error message back (not a Python exception) so it can decide whether to retry, try a different tool, or tell the user it can't complete the task. If you let exceptions bubble up silently, the model loses track of what happened.

### How the Message Loop Works

This is the part that's easiest to get wrong the first time you implement it.

When you call the API with tools defined, the model might respond with `finish_reason: "tool_calls"` instead of `"stop"`. That means it wants to call a tool — you shouldn't return yet. You need to execute the tool and send the result back before making another LLM call.

The full sequence looks like this:

```python
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=TOOL_SCHEMAS,
    tool_choice="auto",
)

if response.choices[0].finish_reason == "tool_calls":
    # 1. Append the assistant message to history (the one containing tool_calls)
    messages.append(response.choices[0].message)

    # 2. Execute each requested tool and append its result
    for tool_call in response.choices[0].message.tool_calls:
        result = execute_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,   # must match — this is how the model knows which result belongs to which call
            "content": json.dumps(result),
        })

    # 3. Call the LLM again — now it has the tool results in context
    response = client.chat.completions.create(model="gpt-4o", messages=messages)
```

The `tool_call_id` is what the API uses to link each result back to the specific call that requested it. If you omit it, the API throws an error. If the model requested two tools at once, you need two separate `tool` messages, each with its own matching ID.

### Parallel tool calling

The model can also request multiple tools in a single response — `message.tool_calls` will have more than one entry. This happens when the model figures out it needs two independent pieces of information and doesn't need to wait for one before fetching the other.

```python
# model returns two tool_calls at once — run them both, append both results
for tool_call in response.choices[0].message.tool_calls:
    result = execute_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})

# one final LLM call with both results now in context
final = client.chat.completions.create(model="gpt-4o", messages=messages)
```

### Things That Catch You Out

Tool descriptions are more important than they look. The model picks tools by reading the description — if two tools have overlapping descriptions, or if the description is vague, the model will occasionally pick the wrong one. The fix is adding "use this when..." phrasing that makes the right choice unambiguous.

Errors need to come back as strings in the message history, not as Python exceptions. If `get_weather` throws an exception and you let it propagate, the model never gets feedback — it just sees an incomplete conversation and may hallucinate a result. Always catch and return `{"error": "..."}`.

Large tool outputs are a real problem. A web search that returns full article text, or a database query returning hundreds of rows, can blow through your context budget in a single tool call. You need to either paginate, truncate, or summarize results before appending them to the message history.

The iteration cap matters. Without it, a model that gets stuck in a tool-calling loop will run indefinitely. Five iterations is a reasonable default for most tasks — if it hasn't found an answer by then, something is wrong with either the tools or the query.

One ordering thing: you must append the assistant's `tool_calls` message before appending the tool results. The API validates that each `tool` message has a matching preceding assistant message with that `tool_call_id`. If you get this backwards, you'll get a validation error.

### Skeleton Implementation

Tool definitions and execution functions — these work the same whether you're using mock or real implementations.

In [ ]:
import json
from typing import Any, Dict, List

# Tool schemas in the format OpenAI expects for the `tools` parameter.
# The description is what the model reads to decide whether to call this tool —
# be specific about when to use it, not just what it does.
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a mathematical expression and return the result. Use this for any arithmetic, percentage, or numerical calculation instead of computing yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to evaluate, e.g. '87.50 * 0.15' or '(100 + 200) / 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather conditions for a specific city. Use this when the user asks about weather, temperature, or conditions in a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'London', 'New York'"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature units. Default is celsius.",
                        "default": "celsius"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for current information on a topic. Use this when the user asks about recent events, news, or anything that may have changed after your training cutoff.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    },
                    "num_results": {
                        "type": "integer",
                        "description": "Number of results to return. Default is 5.",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("Tool schemas loaded:")
for t in TOOL_SCHEMAS:
    print(f"  - {t['function']['name']}: {t['function']['description'][:70]}...")

In [ ]:
import math

def calculator(expression: str) -> Dict[str, Any]:
    """Evaluate a math expression safely."""
    try:
        # Allow math functions but no builtins
        result = eval(expression, {"__builtins__": {}, "math": math}, {})
        return {"success": True, "result": result, "expression": expression}
    except Exception as e:
        return {"success": False, "error": str(e), "expression": expression}


def get_weather(city: str, units: str = "celsius") -> Dict[str, Any]:
    """Get weather for a city (mock — swap for a real API call in production)."""
    mock_data = {
        "New York":  {"temp_c": 22, "condition": "Partly cloudy"},
        "London":    {"temp_c": 15, "condition": "Rainy"},
        "Tokyo":     {"temp_c": 25, "condition": "Sunny"},
        "Paris":     {"temp_c": 18, "condition": "Clear"},
    }

    data = mock_data.get(city)
    if not data:
        return {"success": False, "error": f"No weather data for '{city}'"}

    temp = data["temp_c"] if units == "celsius" else (data["temp_c"] * 9 / 5) + 32
    return {
        "success": True,
        "city": city,
        "temperature": temp,
        "units": units,
        "condition": data["condition"],
    }


def search_web(query: str, num_results: int = 5) -> Dict[str, Any]:
    """Search the web (mock — swap for a real search API in production)."""
    return {
        "success": True,
        "query": query,
        "results": [
            {"title": f"Result {i + 1} for '{query}'", "url": f"https://example.com/{i + 1}"}
            for i in range(num_results)
        ],
    }


# Registry maps tool names (as the model knows them) to Python functions
TOOL_REGISTRY = {
    "calculator": calculator,
    "get_weather": get_weather,
    "search_web": search_web,
}


def execute_tool(tool_name: str, parameters: Dict[str, Any]) -> Dict[str, Any]:
    """
    Execute a tool by name. Always returns a dict — never raises.
    The model needs to see error messages as content, not exceptions.
    """
    if tool_name not in TOOL_REGISTRY:
        return {"success": False, "error": f"Unknown tool: '{tool_name}'"}
    try:
        return TOOL_REGISTRY[tool_name](**parameters)
    except Exception as e:
        return {"success": False, "error": str(e)}


# Quick sanity checks
print("calculator(15 * 8):", execute_tool("calculator", {"expression": "15 * 8"}))
print("get_weather(London):", execute_tool("get_weather", {"city": "London"}))
print("search_web(AI 2026):", execute_tool("search_web", {"query": "AI 2026", "num_results": 2}))

### Working Implementation with Real Function Calling

The skeleton above uses `if/elif` keyword matching to simulate tool selection — that's not how it actually works. Below is the real implementation using the OpenAI `tools` API.

The key difference: the model decides which tool to call based on the descriptions. Your code just executes whatever the model requests and feeds the result back via the message history. The loop continues until `finish_reason == "stop"` (model has a final answer) or we hit max iterations.

In [ ]:
import os
from openai import OpenAI

# Requires OPENAI_API_KEY set in your environment
client = OpenAI()


def agent_with_tools(user_query: str, max_iterations: int = 5) -> str:
    """
    Agent loop using real OpenAI function calling.

    The model decides which tools to call — we don't match keywords.
    Each iteration: call LLM → if tool_calls, execute them and append results → repeat.
    Loop ends when finish_reason is "stop" (model has a final answer).
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant with access to tools. Use them when needed."
        },
        {"role": "user", "content": user_query},
    ]

    for iteration in range(max_iterations):
        print(f"\n[iteration {iteration + 1}] Calling LLM...")

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOL_SCHEMAS,    # defined in next cell
            tool_choice="auto",
        )

        choice = response.choices[0]
        print(f"  finish_reason: {choice.finish_reason}")

        # Model is done — return the final answer
        if choice.finish_reason == "stop":
            return choice.message.content

        # Model wants to call one or more tools
        if choice.finish_reason == "tool_calls":
            # Step 1: append the assistant message (contains the tool_calls list)
            messages.append(choice.message)

            # Step 2: execute each tool and append results
            for tool_call in choice.message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                print(f"  tool call: {name}({args})")

                result = execute_tool(name, args)
                print(f"  result: {result}")

                # Step 3: append the tool result — tool_call_id must match
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                })

    return "Reached max iterations without a final answer."


# Run — requires OPENAI_API_KEY
print("=" * 60)
print("QUERY: What's the weather in London and 15% tip on $87.50?")
print("=" * 60)
answer = agent_with_tools("What's the weather in London right now, and what's a 15% tip on $87.50?")
print(f"\nFINAL ANSWER:\n{answer}")

### Parallel Tool Calling

When the model needs two independent pieces of information, it can request both tools in a single response instead of making two sequential calls. The `tool_calls` list will have multiple entries — you execute all of them, append all results, then call the LLM once more.

The course specifically covers this because it's a meaningful performance win for queries that naturally split into independent lookups (e.g. "compare weather in London and Tokyo" — neither result depends on the other).

In [ ]:
def agent_parallel_tools(user_query: str) -> str:
    """
    Same agent loop, but explicitly handles the case where the model
    requests multiple tools in a single response (parallel tool calling).

    When the model needs two independent pieces of information, it can
    return both tool_calls at once. We execute all of them, append all
    results, then make one final LLM call.
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools."},
        {"role": "user", "content": user_query},
    ]

    for iteration in range(5):
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
        )

        choice = response.choices[0]

        if choice.finish_reason == "stop":
            return choice.message.content

        if choice.finish_reason == "tool_calls":
            # Append the assistant message first (required)
            messages.append(choice.message)

            num_calls = len(choice.message.tool_calls)
            print(f"[iteration {iteration + 1}] Model requested {num_calls} tool call(s) in parallel:")

            # Execute all tool calls and append each result
            for tool_call in choice.message.tool_calls:
                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)
                print(f"  - {name}({args})")
                result = execute_tool(name, args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,   # links result to its specific call
                    "content": json.dumps(result),
                })

    return "Reached max iterations."


# This query should trigger two tool calls in one shot:
# weather for London + weather for Tokyo
print("=" * 60)
print("QUERY: Compare weather in London and Tokyo")
print("=" * 60)
answer = agent_parallel_tools("What's the weather like in London and Tokyo right now?")
print(f"\nFINAL ANSWER:\n{answer}")

### How It Connects to the Other Patterns

Tool use is the most naturally composable of the four patterns.

With reflection, tools provide the external validation feedback — instead of the LLM critiquing its own code, you run the code and feed back the actual errors. The "external feedback" approach from the reflection notebook is essentially just tool use.

With planning, the planner outputs a sequence of steps and tools are how those steps get executed. The model says "I need to look up X, then compute Y using that result" — the planning pattern structures that reasoning, and tool calls carry it out.

With RAG, retrieval becomes a tool rather than a pre-step. Instead of always fetching context upfront, the model decides mid-conversation when it needs more information and calls the retrieval tool. This tends to be more targeted and saves tokens on queries that don't actually need external context.

Multi-agent systems are where tool use really scales — you can give different agents access to different tool sets. A research agent gets search tools, a code agent gets an interpreter, a data agent gets database access. No single agent needs access to everything.

### Notes

The three things that actually matter when building this: write tool descriptions that make the right choice obvious, always return errors as strings so the model can handle them gracefully, and make sure the message history follows the right order (assistant message first, then tool results with matching IDs).

### Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                     USER QUERY                              │
│            "What's 15% tip on $87.50?"                      │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│             LLM (finish_reason: "tool_calls")               │
│                                                             │
│  Decides to call calculator tool                            │
│  tool_calls: [{                                             │
│    id: "call_abc123",                                       │
│    function: {                                              │
│      name: "calculator",                                    │
│      arguments: '{"expression": "87.50 * 0.15"}'           │
│    }                                                        │
│  }]                                                         │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      │  append assistant message to history
                      ▼
┌─────────────────────────────────────────────────────────────┐
│                YOUR CODE EXECUTES THE TOOL                  │
│                                                             │
│  calculator(expression="87.50 * 0.15")                      │
│  → {"success": true, "result": 13.125}                      │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      │  append tool message with tool_call_id
                      ▼
┌─────────────────────────────────────────────────────────────┐
│  messages = [                                               │
│    {"role": "user",      "content": "What's 15% tip..."},   │
│    {"role": "assistant", "tool_calls": [...]},              │  ← must be here
│    {"role": "tool",      "tool_call_id": "call_abc123",     │  ← matching id
│                          "content": '{"result": 13.125}'},  │
│  ]                                                          │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│             LLM (finish_reason: "stop")                     │
│                                                             │
│  Reads the tool result from history, composes answer        │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│                  FINAL RESPONSE                             │
│  "A 15% tip on $87.50 is $13.13. Total would be $100.63."  │
└─────────────────────────────────────────────────────────────┘
```

If the model needs multiple tools, the loop repeats — each iteration adds more tool call + result pairs to the message history before the LLM makes its final pass.